# PMFBY: full-context simulation and prioritisation
**Primary submission notebook.** Run all cells, in order, on Databricks with a Spark session and a writable Unity Catalog schema. Clone/import the complete repository first. No private source tables, tokens or APIs are required.

Inputs: the three bundled v2.1 intermediate snapshots used by the accepted v2.2 experiment. Outputs: new Delta tables, never overwrites. Default workload: 2,000,000 synthetic scenarios. The exact historical results remain separately in `results/`; a new execution is a new run, not a replacement for those results.

The optional `03_rebuild_source_features.ipynb` shows how the intermediate weather/vegetation/master layers were engineered from the bundled source extracts. Start here to reproduce the final experiment without repeating that extraction work.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = None
candidate = Path(PROJECT_ROOT or Path.cwd()).resolve()
root = (
    next((p for p in (candidate,
             *candidate.parents) if (p / 'project_config.json').is_file()),
         None)
)
if root is None:
    raise FileNotFoundError('Import/clone the COMPLETE repository; set PROJECT_ROOT if needed.')
sys.path.insert(0, str(root))
from src.project_runtime import ProjectRuntime
import json
import math
from functools import reduce
from time import perf_counter
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
runtime = ProjectRuntime(spark, root)
VERSION = runtime.config['simulation_version']
assert VERSION == '2.2.0', 'Changing VERSION also changes all generated identifiers.'
RUN_ID, PREFIX = (runtime.run_id, runtime.prefix)
SEED, TARGET = (runtime.config['seed'], runtime.config['synthetic_rows'])
assert TARGET >= 4497
MAX_GRID_DISTANCE_KM, MIN_DAY_SHARE = (75.0, 0.95)
TRAIN_YEARS, VALIDATION_YEAR, TEST_YEAR = ([2018, 2019, 2020], 2021, 2022)
TRAIN_SAMPLE_SHARE = 0.2
CAPACITIES = [0.1, 0.2, 0.3]
SOURCE_VERSIONS, PUBLISHED, TIMINGS = (runtime.source_versions, runtime.published, runtime.timings)
STARTED = runtime.started
publish, table = (runtime.publish, runtime.table)

def unique(df, keys, description):
    assert not df.groupBy(*keys).count().filter('count > 1').limit(1).count(), description

def require(df, columns):
    assert len(df.columns) == len(set(df.columns)), 'Duplicate columns'
    missing = set(columns) - set(df.columns)
    assert not missing, 'Missing columns: ' + repr(sorted(missing))

def uniform(id_col, salt):
    return (F.pmod(F.xxhash64(F.col(id_col), F.lit(SEED), F.lit(salt)), F.lit(1000000000)) + F.lit(0.5)) / F.lit(1000000000.0)

def normalize(column):
    return F.upper(F.regexp_replace(F.trim(column.cast('string')), '[^A-Za-z0-9]', ''))


## 1. Retain and repair all 4,497 contexts
Recover only documented exact names/centroids. Keep original columns intact; put regional median estimates in `_model`, with `_missing` and `_fill_method` provenance. Missing historical anomalies receive flagged neutral model placeholders, not invented observations. Retrospective same-season peer estimates are allowed by this experiment's scope.

In [ ]:
original = runtime.read_snapshot('base_gold')
weather = runtime.read_snapshot('weather_seasonal')
vegetation = runtime.read_snapshot('vegetation_seasonal')
master_n = original.count()
coord = (
    spark.createDataFrame([('MEGHALAYA',
             'RIBHOI',
             25.87043721278977,
             91.8910240541252),
             ('UTTARPRADESH',
             'BARABANKI',
             26.938642333407014,
             81.32740262265608),
             ('UTTARPRADESH',
             'SIDDHARTHNAGAR',
             27.27405417091791,
             82.87344264984435)],
         ['state_key',
             'district_key',
             'reference_lat',
             'reference_lon'])
)
gold = original.join(coord, ['state_key', 'district_key'], 'left')
gold = (
    gold.withColumn('centroid_recovered',
         F.col('centroid_lat').isNull() & F.col('reference_lat').isNotNull())
    .withColumn('centroid_lat',
         F.coalesce('centroid_lat',
             'reference_lat'))
    .withColumn('centroid_lon',
         F.coalesce('centroid_lon',
             'reference_lon'))
    .drop('reference_lat',
         'reference_lon')
)
points = gold.filter('centroid_recovered').select('centroid_lat', 'centroid_lon').distinct()
grids = weather.select('grid_lat', 'grid_lon').distinct()
cross = points.crossJoin(F.broadcast(grids))
a = (
    F.pow(F.sin(F.radians(F.col('grid_lat') - F.col('centroid_lat')) / 2),
         2) + F.cos(F.radians('centroid_lat')) * F.cos(F.radians('grid_lat')) * F.pow(F.sin(F.radians(F.col('grid_lon') - F.col('centroid_lon')) / 2),
         2)
)
nearest = cross.withColumn('new_distance', 2 * 6371.0088 * F.asin(F.sqrt(F.least(F.lit(1.0), a))))
nearest = (
    nearest.withColumn('_rank',
         F.row_number().over(Window.partitionBy('centroid_lat',
             'centroid_lon').orderBy('new_distance',
             'grid_lat',
             'grid_lon')))
    .filter('_rank=1')
    .select('centroid_lat',
         'centroid_lon',
         F.col('grid_lat').alias('new_lat'),
         F.col('grid_lon').alias('new_lon'),
         'new_distance')
)
gold = (
    gold.join(nearest,
         ['centroid_lat',
             'centroid_lon'],
         'left')
    .withColumn('grid_lat',
         F.coalesce('grid_lat',
             'new_lat'))
    .withColumn('grid_lon',
         F.coalesce('grid_lon',
             'new_lon'))
    .withColumn('weather_grid_distance_km',
         F.coalesce('weather_grid_distance_km',
             'new_distance'))
    .drop('new_lat',
         'new_lon',
         'new_distance')
)
keys = ['grid_lat', 'grid_lon', 'year', 'season']
wcols = [c for c in weather.columns if c not in keys]
gold = gold.drop(*wcols).join(weather, keys, 'left')
gold = (
    gold.withColumn('_veg_state',
         F.when(F.col('state_key') == 'ODISHA',
             'ORISSA').when(F.col('state_key') == 'TELANGANA',
             'ANDHRAPRADESH').otherwise(F.col('state_key')))
    .withColumn('_veg_district',
         F.coalesce(normalize(F.col('district_2011_raw')),
             F.col('district_key')))
)
vcols = ['ndvi_mean', 'evi_mean', 'ndvi_anomaly', 'ndvi_prior_mean', 'ndvi_history_years']
v = (
    vegetation.select(F.col('state_key').alias('_veg_state'),
         F.col('district_key').alias('_veg_district'),
         'year',
         'season',
         *[F.col(c).alias('_recovered_' + c) for c in vcols])
)
gold = gold.join(v, ['_veg_state', '_veg_district', 'year', 'season'], 'left')
gold = (
    gold.withColumn('vegetation_recovered',
         F.col('ndvi_mean').isNull() & F.col('_recovered_ndvi_mean').isNotNull())
)
for c in vcols:
    gold = (
        gold.withColumn(c,
             F.coalesce(F.col(c),
                 F.col('_recovered_' + c)))
        .drop('_recovered_' + c)
    )
gold = gold.drop('_veg_state', '_veg_district')
FEATURES_BASE = (
    ['coverage_avg_premium_rate',
         'coverage_n_crops',
         'weather_temp_mean_c',
         'weather_precip_anomaly_pct',
         'ndvi_mean',
         'evi_mean',
         'ndvi_anomaly',
         'weather_valid_day_share']
)
FEATURES_EVENTS = (
    ['weather_temp_max_daily_mean_c',
         'weather_dry_days_lt_1mm',
         'weather_heat_days_mean_ge_35c',
         'weather_heavy_rain_days_ge_20mm',
         'weather_heavy_rain_days_ge_50mm',
         'weather_max_consecutive_dry_days',
         'weather_max_daily_precip_mm',
         'weather_max_3day_precip_mm']
)
RAW_FEATURES = FEATURES_BASE + FEATURES_EVENTS
non_anomaly = (
    [c for c in RAW_FEATURES if c not in ['ndvi_anomaly',
             'weather_precip_anomaly_pct',
             'weather_valid_day_share']]
)
state_stats = (
    gold.groupBy('state_key',
         'year',
         'season')
    .agg(*[F.percentile_approx(c,
             0.5).alias('_state_' + c) for c in non_anomaly])
)
season_stats = (
    gold.groupBy('year',
         'season')
    .agg(*[F.percentile_approx(c,
             0.5).alias('_season_' + c) for c in non_anomaly])
)
gold = (
    gold.join(state_stats,
         ['state_key',
             'year',
             'season'],
         'left')
    .join(season_stats,
         ['year',
             'season'],
         'left')
)
for c in non_anomaly:
    observed = F.col(c).isNotNull() & ~F.isnan(F.col(c).cast('double'))
    gold = (
        gold.withColumn(c + '_model',
             F.coalesce(F.when(observed,
                 F.col(c)),
                 F.col('_state_' + c),
                 F.col('_season_' + c)).cast('double'))
        .withColumn(c + '_fill_method',
             F.when(observed,
                 'observed_or_exact_recovery').when(F.col('_state_' + c).isNotNull(),
                 'state_year_season_median').otherwise('year_season_median'))
        .withColumn(c + '_missing',
             (~observed).cast('double'))
        .drop('_state_' + c,
             '_season_' + c)
    )
for c in ['ndvi_anomaly', 'weather_precip_anomaly_pct']:
    gold = (
        gold.withColumn(c + '_model',
             F.coalesce(F.col(c),
                 F.lit(0.0)).cast('double'))
        .withColumn(c + '_missing',
             F.col(c).isNull().cast('double'))
        .withColumn(c + '_fill_method',
             F.when(F.col(c).isNotNull(),
                 'observed_prior_year_anomaly').otherwise('no_prior_baseline_neutral_placeholder'))
    )
c = 'weather_valid_day_share'
gold = (
    gold.withColumn(c + '_model',
         F.coalesce(F.col(c),
             F.lit(0.0)).cast('double'))
    .withColumn(c + '_missing',
         F.col(c).isNull().cast('double'))
    .withColumn(c + '_fill_method',
         F.when(F.col(c).isNotNull(),
             'observed_completeness').otherwise('no_local_weather_zero_completeness'))
)
gold = (
    gold.withColumn('feature_imputation_count',
         sum([F.col(c + '_missing') for c in RAW_FEATURES]))
    .withColumn('evidence_tier',
         F.when(F.col('feature_imputation_count') == 0,
             'observed_complete').when(F.col('centroid_lat').isNull() | F.col('ndvi_mean').isNull(),
             'regional_fallback').otherwise('limited_history'))
    .withColumn('previous_model_ready',
         F.col('model_ready_after_external_features'))
    .withColumn('model_ready_after_external_features',
         F.lit(True))
    .withColumn('feature_schema_version',
         F.lit(VERSION))
)
unique(gold, ['context_key'], 'Master context fan-out')
assert gold.count() == master_n == 4497
assert gold.dropna(subset=[c + '_model' for c in RAW_FEATURES]).count() == master_n
assert gold.filter('coverage_sum_insured_total <= 0').count() == 0
publish(gold, 'gold', ['year', 'season'])
coverage = (
    gold.groupBy('year',
         'season')
    .agg(F.count('*').alias('retained_contexts'),
         F.sum(F.col('previous_model_ready').cast('int')).alias('previous_ready'),
         F.sum(F.col('vegetation_recovered').cast('int')).alias('vegetation_recovered'),
         F.sum(F.col('centroid_recovered').cast('int')).alias('centroid_recovered'))
)
publish(coverage, 'coverage')
fill_audit = (
    reduce(lambda a,
         b: a.unionByName(b),
         [gold.groupBy(c + '_fill_method').count().select(F.lit(c).alias('feature'),
             F.col(c + '_fill_method').alias('method'),
             'count') for c in RAW_FEATURES])
)
publish(fill_audit, 'fill_audit')
(
    print('CONTEXT_GATE_PASSED',
         json.dumps({'retained': gold.count(),
             'previous': 1719,
             'coverage': [r.asDict() for r in coverage.orderBy('year', 'season').collect()],
             'tiers': [r.asDict() for r in gold.groupBy('evidence_tier').count().collect()]}),
         flush=True)
)


## 2. Generate two million scenarios
Give every context at least one scenario. Allocate the remainder half uniformly and half by insured exposure, resolving fractional allocation deterministically. This is a scenario design, not a farmer population estimate.

The uncalibrated simulator begins at 7%, adds documented weather/vegetation contributions and a uniform 0–0.12 shock, clips to [0.02,0.95], and samples a separate binary event. Both losses and non-losses occur. Observable scenarios, hidden truth and labels are separate physical tables. Never include hidden truth in model features or policy ranking.

In [ ]:
FEATURES = RAW_FEATURES + [c + '_missing' for c in RAW_FEATURES]
eligible = gold
for c in RAW_FEATURES:
    eligible = eligible.withColumn(c, F.col(c + '_model'))
require(eligible, FEATURES)
assert eligible.dropna(subset=FEATURES).count() == eligible.count(), 'Readiness does not cover model feature contract'
assert eligible.filter(F.col('year').isin(TRAIN_YEARS)).count() > 0 and eligible.filter(F.col('year') == TEST_YEAR).count() > 0
allocation = (
    eligible.withColumn('_expected',
         1 + (TARGET - master_n) * (0.5 / F.lit(master_n) + 0.5 * F.col('coverage_sum_insured_total') / F.sum('coverage_sum_insured_total').over(Window.partitionBy())))
    .withColumn('scenario_count',
         F.floor('_expected').cast('long'))
    .withColumn('_fraction',
         F.col('_expected') - F.col('scenario_count'))
)
shortfall = TARGET - allocation.agg(F.sum('scenario_count')).first()[0]
allocation = (
    allocation.withColumn('_r',
         F.row_number().over(Window.orderBy(F.desc('_fraction'),
             'context_key')))
    .withColumn('scenario_count',
         F.col('scenario_count') + F.when(F.col('_r') <= shortfall,
             1).otherwise(0))
    .drop('_r',
         '_expected',
         '_fraction')
)
assert allocation.agg(F.sum('scenario_count')).first()[0] == TARGET
allocation = (
    publish(allocation.select('context_key',
             'year',
             'season',
             'coverage_sum_insured_total',
             'scenario_count'),
         'allocation')
)
scenarios = (
    eligible.join(allocation.select('context_key',
             'scenario_count'),
         'context_key')
    .filter('scenario_count > 0')
    .withColumn('scenario_sequence',
         F.explode(F.sequence(F.lit(1),
             F.col('scenario_count').cast('int'))))
    .withColumn('synthetic_claim_id',
         F.sha2(F.concat_ws('|',
             F.lit(VERSION),
             'context_key',
             'scenario_sequence',
             F.lit(SEED)),
             256))
    .withColumn('synthetic_insured_amount_proxy',
         F.col('coverage_sum_insured_total') / F.col('scenario_count') * (0.75 + 0.5 * uniform('synthetic_claim_id',
             'amount')))
    .withColumn('synthetic_record',
         F.lit(True))
    .withColumn('simulation_version',
         F.lit(VERSION))
)
probability = (
    F.lit(0.07) + F.when(F.col('weather_precip_anomaly_pct') <= -30,
         0.22)
    .when(F.col('weather_precip_anomaly_pct') <= -10,
         0.1)
    .otherwise(0.0) + F.when(F.col('ndvi_anomaly') <= -0.15,
         0.2)
    .when(F.col('ndvi_anomaly') <= -0.05,
         0.08)
    .otherwise(0.0) + F.when(F.col('weather_temp_max_daily_mean_c') >= 30,
         0.08)
    .otherwise(0.0) + F.when(F.col('weather_heavy_rain_days_ge_20mm') >= 3,
         0.05)
    .otherwise(0.0) + uniform('synthetic_claim_id',
         'unobserved_shock') * 0.12
)
truth = (
    scenarios.select('synthetic_claim_id',
         F.least(F.lit(0.95),
             F.greatest(F.lit(0.02),
             probability)).alias('hidden_loss_probability'))
    .withColumn('simulated_loss_event',
         (uniform('synthetic_claim_id',
             'loss_event') < F.col('hidden_loss_probability')).cast('int'))
    .withColumn('synthetic_record',
         F.lit(True))
)
truth = publish(truth, 'hidden_truth')
labels = (
    publish(truth.select('synthetic_claim_id',
             F.col('simulated_loss_event').cast('double').alias('label'),
             'synthetic_record'),
         'labels')
)
OBSERVABLE_COLUMNS = (
    ['synthetic_claim_id',
         'context_key',
         'district_censuscode',
         'state',
         'year',
         'season',
         'coverage_sum_insured_total',
         'synthetic_insured_amount_proxy',
         'synthetic_record',
         'simulation_version',
         'evidence_tier',
         'feature_imputation_count'] + FEATURES
)
claims = publish(scenarios.select(*OBSERVABLE_COLUMNS), 'synthetic_claims', ['year', 'season'])
assert claims.count() == TARGET and labels.count() == TARGET and (truth.count() == TARGET)
unique(claims, ['synthetic_claim_id'], 'Non-unique scenario IDs')
assert not any(('hidden' in c or c in ['label', 'simulated_loss_event'] for c in claims.columns))
assert claims.join(labels, 'synthetic_claim_id', 'left_anti').count() == 0
assert claims.select('context_key').distinct().count() == master_n
print('SYNTHETIC_STAGE_COMPLETE', TARGET, 'CONTEXTS', master_n, flush=True)


## 3. Train and evaluate three model specifications
Logistic regression and two random forests use a deterministic 20% sample of 2018–2020 for training. Select classification thresholds on 2021 only; score the complete 2022 cohort. Full features are 16 numeric values plus 16 missingness flags. The baseline RF uses the eight base features and their flags. Models receive context evidence, so cases sharing that context receive identical scores.

RF: 60 trees, depth 8. LR: 60 iterations, regularisation 0.01. These were fixed engineering/compute choices, not results of a hyperparameter search. MLflow model logging is optional and disabled by default to avoid workspace-specific tracking dependencies; predictions and metrics always persist.

In [ ]:
train_x = (
    claims.filter(F.col('year').isin(TRAIN_YEARS))
    .filter(uniform('synthetic_claim_id',
             'train_sample') < TRAIN_SAMPLE_SHARE)
)
test_x = claims.filter(F.col('year') == TEST_YEAR)
train = train_x.join(labels.select('synthetic_claim_id', 'label'), 'synthetic_claim_id')
test_labels = (
    labels.join(test_x.select('synthetic_claim_id'),
         'synthetic_claim_id')
    .select('synthetic_claim_id',
         'label')
)
train_n, test_n = (train.count(), test_x.count())
assert train.select('label').distinct().count() == 2 and test_labels.select('label').distinct().count() == 2
assert train_x.select('context_key').distinct().join(test_x.select('context_key').distinct(), 'context_key').count() == 0
validation_x = claims.filter(F.col('year') == VALIDATION_YEAR)
validation_labels = labels.select('synthetic_claim_id', 'label')
threshold_rows = []
metrics_rows, class_rows, importance_rows, predictions_all = ([], [], [], [])
MODEL_SPECS = (
    [('logistic_regression',
             FEATURES,
             'lr'),
         ('random_forest',
             FEATURES,
             'rf'),
         ('random_forest_base_features',
             FEATURES_BASE + [c + '_missing' for c in FEATURES_BASE],
             'rf')]
)
for model_name, feature_names, kind in MODEL_SPECS:
    t = perf_counter()
    assembler = (
        VectorAssembler(inputCols=feature_names,
             outputCol='assembled',
             handleInvalid='error')
    )
    scaler = (
        StandardScaler(inputCol='assembled',
             outputCol='features',
             withMean=False,
             withStd=True)
    )
    estimator = (
        LogisticRegression(maxIter=60,
             regParam=0.01,
             labelCol='label') if kind == 'lr' else RandomForestClassifier(numTrees=60,
             maxDepth=8,
             seed=SEED,
             labelCol='label')
    )
    fitted = (
        Pipeline(stages=[assembler,
                 scaler,
                 estimator])
        .fit(train.select(*feature_names + ['label']))
    )
    fit_seconds = perf_counter() - t
    out = (
        fitted.transform(test_x)
        .select('synthetic_claim_id',
             vector_to_array('probability')[1].alias('predicted_loss_risk'))
        .withColumn('model',
             F.lit(model_name))
    )
    out = publish(out, 'predictions_' + model_name)
    predictions_all.append(out)
    evaluated = out.join(test_labels, 'synthetic_claim_id')
    roc = (
        BinaryClassificationEvaluator(labelCol='label',
             rawPredictionCol='predicted_loss_risk',
             metricName='areaUnderROC')
        .evaluate(evaluated)
    )
    pr = (
        BinaryClassificationEvaluator(labelCol='label',
             rawPredictionCol='predicted_loss_risk',
             metricName='areaUnderPR')
        .evaluate(evaluated)
    )
    val = (
        fitted.transform(validation_x)
        .select('synthetic_claim_id',
             vector_to_array('probability')[1].alias('risk'))
        .join(validation_labels,
             'synthetic_claim_id')
    )
    thresholds = spark.createDataFrame([(i / 100.0,) for i in range(5, 66, 5)], ['threshold'])
    candidates = (
        val.crossJoin(F.broadcast(thresholds))
        .groupBy('threshold')
        .agg(F.sum(F.when((F.col('risk') >= F.col('threshold')) & (F.col('label') == 1),
                 1).otherwise(0)).alias('tp'),
             F.sum(F.when((F.col('risk') >= F.col('threshold')) & (F.col('label') == 0),
                 1).otherwise(0)).alias('fp'),
             F.sum(F.when((F.col('risk') < F.col('threshold')) & (F.col('label') == 1),
                 1).otherwise(0)).alias('fn'))
    )
    chosen = (
        candidates.withColumn('f1',
             2 * F.col('tp') / (2 * F.col('tp') + F.col('fp') + F.col('fn')))
        .orderBy(F.desc('f1'),
             F.desc('threshold'))
        .first()
    )
    threshold = float(chosen.threshold)
    threshold_rows.append((model_name, threshold, float(chosen.f1)))
    cm = (
        evaluated.withColumn('prediction',
             (F.col('predicted_loss_risk') >= threshold).cast('double'))
        .groupBy('label',
             'prediction')
        .count()
        .collect()
    )
    counts = {(int(r.label), int(r.prediction)): r['count'] for r in cm}
    f1s, supports = ([], [])
    for label in [0, 1]:
        tp = counts.get((label, label), 0)
        fp = counts.get((1 - label, label), 0)
        fn = counts.get((label, 1 - label), 0)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        class_rows.append((model_name, label, tp + fn, tp, fp, fn, precision, recall, f1))
        f1s.append(f1)
        supports.append(tp + fn)
    brier = evaluated.agg(F.avg(F.pow(F.col('predicted_loss_risk') - F.col('label'), 2))).first()[0]
    (
        metrics_rows.append((model_name,
                 float(roc),
                 float(pr),
                 sum(f1s) / 2,
                 sum((f * s for f, s in zip(f1s, supports))) / sum(supports),
                 float(brier),
                 train_n,
                 test_n,
                 float(fit_seconds)))
    )
    if kind == 'rf':
        (
            importance_rows.extend(((model_name,
                     c,
                     float(v)) for c,
                     v in zip(feature_names,
                     fitted.stages[-1].featureImportances)))
        )
    if runtime.config['log_models_to_mlflow']:
        import os
        import mlflow
        import mlflow.spark
        volume = runtime.namespace + '.pmfby_submission_models'
        spark.sql('CREATE VOLUME IF NOT EXISTS ' + volume)
        os.environ['MLFLOW_DFS_TMP'] = (
            '/Volumes/' + runtime.namespace.replace('.',
                 '/') + '/pmfby_submission_models/' + RUN_ID
        )
        dbutils.fs.mkdirs(os.environ['MLFLOW_DFS_TMP'])
        with mlflow.start_run(run_name=RUN_ID + '_' + model_name):
            (
                mlflow.log_params({'simulation_version': VERSION,
                         'seed': SEED,
                         'features': json.dumps(feature_names)})
            )
            (
                mlflow.log_metrics({'synthetic_roc_auc': roc,
                         'synthetic_pr_auc': pr,
                         'synthetic_brier': float(brier)})
            )
            mlflow.spark.log_model(fitted, 'model')
    print('MODEL_COMPLETE', model_name, json.dumps(metrics_rows[-1]), flush=True)
predictions = publish(reduce(lambda a, b: a.unionByName(b), predictions_all), 'model_predictions')
(
    publish(spark.createDataFrame(metrics_rows,
             ['model',
             'roc_auc',
             'pr_auc',
             'macro_f1',
             'weighted_f1',
             'brier_score',
             'training_rows',
             'test_rows',
             'fit_seconds']),
         'model_metrics')
)
(
    publish(spark.createDataFrame(class_rows,
             ['model',
             'label',
             'support',
             'true_positive',
             'false_positive',
             'false_negative',
             'precision',
             'recall',
             'f1']),
         'class_metrics')
)
(
    publish(spark.createDataFrame(importance_rows,
             ['model',
             'feature',
             'importance']),
         'feature_importance')
)
calibration = (
    predictions.join(test_labels,
         'synthetic_claim_id')
    .withColumn('risk_bin',
         F.least(F.lit(9),
             F.floor(F.col('predicted_loss_risk') * 10)))
    .groupBy('model',
         'risk_bin')
    .agg(F.count('*').alias('n'),
         F.avg('predicted_loss_risk').alias('mean_predicted'),
         F.avg('label').alias('observed_synthetic_loss_share'))
)
publish(calibration, 'calibration')
(
    publish(spark.createDataFrame(threshold_rows,
             ['model',
             'validation_selected_threshold',
             'validation_positive_f1']),
         'validation_thresholds')
)


## 4. Compare queues at equal capacity
All six strategies receive the same 10%, 20% and 30% case-count budgets. Scores depend only on predictions or observable evidence. Persist decisions before joining evaluation labels. Ties use scenario ID, not hidden risk. Review routes never approve or deny insurance. Equal review cost and perfect discovery during verification are simulation assumptions.

In [ ]:
weather_score = (
    F.when(F.col('weather_precip_anomaly_pct') <= -10,
         1.0)
    .otherwise(0.0) + F.when(F.col('ndvi_anomaly') <= -0.05,
         1.0)
    .otherwise(0.0) + F.when(F.col('weather_temp_max_daily_mean_c') >= 30,
         1.0)
    .otherwise(0.0)
)
score_frames = (
    [test_x.select('synthetic_claim_id',
             uniform('synthetic_claim_id',
             'random_policy').alias('score')).withColumn('policy',
             F.lit('random')),
         test_x.select('synthetic_claim_id',
             weather_score.alias('score')).withColumn('policy',
             F.lit('weather_vegetation_rule')),
         test_x.select('synthetic_claim_id',
             F.col('synthetic_insured_amount_proxy').alias('score')).withColumn('policy',
             F.lit('exposure_proxy')),
         predictions.select('synthetic_claim_id',
             F.col('predicted_loss_risk').alias('score'),
             F.col('model').alias('policy'))]
)
scores = reduce(lambda a, b: a.unionByName(b), score_frames)
require(scores, ['synthetic_claim_id', 'score', 'policy'])
assert set(scores.columns) == {'synthetic_claim_id', 'score', 'policy'}, 'Policy input contract leaked truth'
ranked = (
    scores.withColumn('priority_rank',
         F.row_number().over(Window.partitionBy('policy').orderBy(F.desc('score'),
             'synthetic_claim_id')))
)
ranked = publish(ranked, 'policy_rankings')
capacity_df = (
    spark.createDataFrame([(float(c),
             int(math.floor(test_n * c))) for c in CAPACITIES],
         ['capacity_share',
             'verification_budget'])
)
decisions = (
    ranked.crossJoin(F.broadcast(capacity_df))
    .withColumn('selected_for_verification',
         F.col('priority_rank') <= F.col('verification_budget'))
)
decisions = publish(decisions, 'policy_decisions', ['policy'])
assert not set(decisions.columns) & {'hidden_loss_probability', 'simulated_loss_event', 'label'}
selection_check = (
    decisions.groupBy('policy',
         'capacity_share',
         'verification_budget')
    .agg(F.sum(F.col('selected_for_verification').cast('long')).alias('selected'))
)
assert selection_check.filter('selected != verification_budget').count() == 0, 'Unequal capacity'
evaluation = decisions.join(test_labels, 'synthetic_claim_id')
summary = (
    evaluation.groupBy('policy',
         'capacity_share',
         'verification_budget')
    .agg(F.count('*').alias('candidates'),
         F.sum('label').alias('total_simulated_losses'),
         F.sum(F.when(F.col('selected_for_verification'),
             F.col('label')).otherwise(0.0)).alias('losses_in_priority_queue'))
    .withColumn('simulated_loss_recall',
         F.col('losses_in_priority_queue') / F.col('total_simulated_losses'))
    .withColumn('simulated_loss_precision',
         F.col('losses_in_priority_queue') / F.col('verification_budget'))
    .withColumn('simulated_missed_loss_share',
         1 - F.col('simulated_loss_recall'))
    .withColumn('interpretation',
         F.lit('same 2022 scenarios; fixed case-count capacity; simulation only'))
)
publish(summary, 'capacity_comparison')
rf_rank = (
    ranked.filter("policy='random_forest'")
    .select('synthetic_claim_id',
         'priority_rank',
         F.col('score').alias('predicted_loss_risk'))
)
routes = (
    test_x.select('synthetic_claim_id',
         'context_key',
         'year',
         'season',
         'weather_valid_day_share')
    .join(rf_rank,
         'synthetic_claim_id')
    .withColumn('triage_route',
         F.when(F.col('priority_rank') <= int(test_n * 0.2),
             'priority_verification').when((F.col('predicted_loss_risk') <= 0.15) & (F.col('weather_valid_day_share') >= 0.99),
             'fast_track_evidence_review').otherwise('standard_review'))
    .withColumn('human_decision_required',
         F.lit(True))
)
publish(routes, 'triage_routes')
publish(routes.groupBy('triage_route').count(), 'triage_summary')
group_eval = (
    evaluation.join(test_x.select('synthetic_claim_id',
             'state',
             'season'),
         'synthetic_claim_id')
    .groupBy('policy',
         'capacity_share',
         'state',
         'season')
    .agg(F.count('*').alias('n'),
         F.sum('label').alias('losses'),
         F.sum(F.when(F.col('selected_for_verification'),
             F.col('label')).otherwise(0.0)).alias('losses_in_queue'))
)
publish(group_eval, 'group_diagnostics')


## 5. Diagnostic, acceptance checks and completion
The simulator's expected-probability diagnostic is evaluation-only. It is not an operational ranking strategy or a proven finite-sample AUC ceiling. A successful run passes every check below and writes `artifacts/latest_run.json`. Do not call a failed/partial run complete. Re-run in a new namespace instead of overwriting old tables.

In [ ]:
expected = (
    test_x.select('synthetic_claim_id',
         (F.lit(0.07) + F.when(F.col('weather_precip_anomaly_pct') <= -30,
             0.22).when(F.col('weather_precip_anomaly_pct') <= -10,
             0.1).otherwise(0.0) + F.when(F.col('ndvi_anomaly') <= -0.15,
             0.2).when(F.col('ndvi_anomaly') <= -0.05,
             0.08).otherwise(0.0) + F.when(F.col('weather_temp_max_daily_mean_c') >= 30,
             0.08).otherwise(0.0) + F.when(F.col('weather_heavy_rain_days_ge_20mm') >= 3,
             0.05).otherwise(0.0) + F.lit(0.06)).alias('expected_probability'))
)
learnability = expected.join(test_labels, 'synthetic_claim_id')
expected_probability_auc = (
    BinaryClassificationEvaluator(labelCol='label',
         rawPredictionCol='expected_probability',
         metricName='areaUnderROC')
    .evaluate(learnability)
)
noise = (
    learnability.agg(F.avg(F.col('expected_probability') * (1 - F.col('expected_probability'))).alias('irreducible_expected_brier'))
    .first()[0]
)
(
    publish(spark.createDataFrame([(float(expected_probability_auc), float(noise), 'known simulator diagnostic; excluded from policy competition')],
             ['simulator_expected_probability_auc',
             'irreducible_expected_brier',
             'interpretation']),
         'learnability_diagnostic')
)
checks = (
    [('all_master_contexts_in_gold',
             gold.count() == 4497),
         ('all_contexts_in_synthetic',
             claims.select('context_key').distinct().count() == 4497),
         ('two_million_unique',
             claims.count() == TARGET and claims.select('synthetic_claim_id').distinct().count() == TARGET),
         ('no_null_model_features',
             claims.dropna(subset=FEATURES).count() == TARGET),
         ('no_truth_in_predictions',
             set(predictions.columns) == {'synthetic_claim_id',
             'predicted_loss_risk',
             'model'}),
         ('no_truth_in_decisions',
             not bool(set(decisions.columns) & {'label', 'hidden_loss_probability', 'simulated_loss_event'})),
         ('equal_capacity',
             selection_check.filter('selected != verification_budget').count() == 0),
         ('full_test_each_model',
             predictions.groupBy('model').count().filter(F.col('count') != test_n).count() == 0),
         ('train_validation_test_disjoint',
             max(TRAIN_YEARS) < VALIDATION_YEAR < TEST_YEAR),
         ('one_label_per_scenario',
             labels.select('synthetic_claim_id').distinct().count() == TARGET)]
)
assert all((ok for _, ok in checks)), checks
publish(spark.createDataFrame(checks, ['check', 'passed']), 'acceptance_checks')
(
    print('V22_FINAL',
         json.dumps({'run_id': RUN_ID,
             'prefix': PREFIX,
             'master_contexts': master_n,
             'synthetic_rows': TARGET,
             'train_rows': train_n,
             'test_rows': test_n,
             'acceptance_passed': len(checks),
             'simulator_expected_probability_auc': expected_probability_auc,
             'irreducible_expected_brier': noise,
             'thresholds': threshold_rows}),
         flush=True)
)
result = (
    runtime.record_completion({'simulation_version': VERSION,
             'seed': SEED,
             'synthetic_rows': TARGET,
             'master_contexts': master_n,
             'training_years': TRAIN_YEARS,
             'validation_year': VALIDATION_YEAR,
             'test_year': TEST_YEAR,
             'model_artifacts_logged': runtime.config['log_models_to_mlflow'],
             'truth_excluded_from_ranking': True,
             'packaging_version': '1.0.0'})
)
(
    publish(spark.createDataFrame([(RUN_ID, json.dumps(result, sort_keys=True))],
             ['run_id',
             'manifest_json']),
         'run_manifest')
)
(
    publish(spark.createDataFrame(list(TIMINGS),
             ['stage',
             'write_and_count_seconds',
             'rows']),
         'stage_timings')
)
print('PIPELINE_COMPLETE', json.dumps(result, sort_keys=True))
display(spark.table(table('model_metrics')))
display(spark.table(table('capacity_comparison')).orderBy('capacity_share', 'policy'))
